Jalankan ini dulu sebelum mengerjakan latihan dibawah.

In [1]:
import os
from pyspark.sql import SparkSession
from pyspark.sql.functions import col

# Membuat SparkSession baru.
spark = SparkSession.builder.appName("Latihan4").master("local[*]").getOrCreate()
spark.sparkContext.setLogLevel("ERROR")

# Menjembatani csv agar tidak mengarah ke HDFS.
jalur_lokal = "file://" + os.path.abspath("data_transaksi_ecommerce.csv")
df = spark.read.csv(jalur_lokal, header=True, inferSchema=True)

# Menambah kolom total pendapatan.
df = df.withColumn("total_pendapatan", col("unit_terjual") * col("harga_satuan"))
print("Siap. Jumlah baris:", df.count())

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/09/15 12:24:32 WARN Utils: Your hostname, caitlyn-ThinkPad-E570, resolves to a loopback address: 127.0.1.1; using 10.55.95.23 instead (on interface wlp5s0)
26/09/15 12:24:32 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/09/15 12:24:33 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/09/15 12:24:34 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


Siap. Jumlah baris: 600


Soal 1. Tampilkan hanya kolom order_id, kategori, dan total_pendapatan untuk transaksi dengan metode_pembayaran bernilai "E-Wallet".

In [2]:
# Menampilkan order_id, kategori, dan total_pendapatan untuk E-Wallet.
df.filter(col("metode_pembayaran") == "E-Wallet") \
  .select("order_id", "kategori", "total_pendapatan") \
  .show(5)

+--------+--------------------+----------------+
|order_id|            kategori|total_pendapatan|
+--------+--------------------+----------------+
|ORD-1000|Kesehatan & Kecan...|         2250000|
|ORD-1003|          Elektronik|          150000|
|ORD-1006|   Makanan & Minuman|           25000|
|ORD-1013|   Makanan & Minuman|          900000|
|ORD-1014|             Fashion|           75000|
+--------+--------------------+----------------+
only showing top 5 rows


Soal 2. Hitung total pendapatan per kategori (bukan per kota), urutkan dari yang tertinggi.

In [3]:
from pyspark.sql.functions import sum as spark_sum

# Menghitung total pendapatan per kategori, urutkan dari yang tertinggi.
df.groupBy("kategori") \
  .agg(spark_sum("total_pendapatan").alias("total_pendapatan")) \
  .orderBy(col("total_pendapatan").desc()) \
  .show()

+--------------------+----------------+
|            kategori|total_pendapatan|
+--------------------+----------------+
|             Fashion|       124825000|
|          Elektronik|       110600000|
|   Makanan & Minuman|        95400000|
|Kesehatan & Kecan...|        82200000|
|        Rumah Tangga|        77350000|
+--------------------+----------------+



Soal 3. Tampilkan jumlah transaksi untuk masing-masing metode_pembayaran (tidak pakai agg).

In [4]:
# Menghitung jumlah transaksi menggunakan shorthand .count()
df.groupBy("metode_pembayaran").count().show()

+-----------------+-----+
|metode_pembayaran|count|
+-----------------+-----+
|              COD|  114|
|    Transfer Bank|  208|
|     Kartu Kredit|   85|
|         E-Wallet|  193|
+-----------------+-----+



Soal 4 (Refleksi singkat). Dalam 2-3 kalimat: apa yang dimaksud dengan lazy evaluation di PySpark, dan mengapa hal ini menguntungkan ketika bekerja dengan data berskala besar? Tulis jawaban pada markdown cell di bawah ini.

Lazy evaluation berarti PySpark tidak langsung mengeksekusi kode transformasi (seperti filter atau select), melainkan hanya mencatatnya ke dalam rencana eksekusi (DAG / Directed Acyclic Graph). Eksekusi baru benar-benar dijalankan saat kita memanggil action (seperti show atau count). Hal ini sangat menguntungkan untuk data berskala besar karena Spark bisa mengoptimalkan seluruh alur pemrosesan dari awal hingga akhir sebelum benar-benar memakan memori komputasi.